In [ ]:
#Part 1 model framework
import torch
import torch.nn as nn

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

class CNNSmoothing(nn.Module):
    def __init__(self, dim=2):
        super().__init__()
        assert dim in (2, 3)
        self.features = nn.Sequential(
            nn.Conv2d(dim, 64, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(128, 1024, kernel_size=1),
            nn.ReLU(),
        )
        self.pool = nn.MaxPool2d(kernel_size=(20, 1))
        self.regressor = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, dim),
        )

    def forward(self, x):
        B, N, D = x.shape
        assert N == 20
        # (B, 20, dim) -> (B, dim, 20, 1)
        x = x.transpose(1, 2).unsqueeze(-1)
        # (B, dim, 20, 1) -> (B, 1024, 20, 1)
        x = self.features(x)
        # (B, 1024, 20, 1) -> (B, 1024)
        x = self.pool(x).flatten(1)
        # (B, 1024) -> (B, dim)
        return self.regressor(x)


model_2d = CNNSmoothing(dim=2)
model_3d = CNNSmoothing(dim=3)
print(model_2d)
print(model_3d)


Using mps device
CNNSmoothing(
  (features): Sequential(
    (0): Conv2d(2, 64, kernel_size=(1, 1), stride=(1, 1))
    (1): ReLU()
    (2): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
    (3): ReLU()
    (4): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
    (5): ReLU()
    (6): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1))
    (7): ReLU()
    (8): Conv2d(128, 1024, kernel_size=(1, 1), stride=(1, 1))
    (9): ReLU()
  )
  (pool): MaxPool2d(kernel_size=(20, 1), stride=(20, 1), padding=0, dilation=1, ceil_mode=False)
  (regressor): Sequential(
    (0): Linear(in_features=1024, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=2, bias=True)
  )
)
CNNSmoothing(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(1, 1), stride=(1, 1))
    (1): ReLU()
    (2): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
    (3): ReLU()
    (4): Conv2d(64, 64, k

In [ ]:
#Part 2 Date generation
